In [54]:
print("hi")

hi


In [55]:
pip install yfinance pandas numpy requests matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Load Phase 1 data

In [56]:
# index_col tells pandas to use the first column as the index of the DataFrame, which is useful for time series data where the first column often contains dates. 
# parse_dates tells pandas to convert the index column to datetime objects instead of leaving them as strings, which allows for easier manipulation and analysis of time series data.

import pandas as pd
import numpy as np
import yfinance as yf

prices = pd.read_csv(
    "../data/raw/prices.csv",
    index_col=0,
    parse_dates=True
)

market = pd.read_csv(
    "../data/raw/market.csv",
    index_col=0,
    parse_dates=True
)

print(prices)
print(market.head())

                  AAPL        AMZN       GOOGL         JPM        META  \
Date                                                                     
2020-01-02   72.271545   94.900497   67.832504  117.899193  207.953827   
2020-01-03   71.568916   93.748497   67.477661  116.343338  206.853500   
2020-01-06   72.139206   95.143997   69.276215  116.250832  210.749298   
2020-01-07   71.799904   95.343002   69.142403  114.274513  211.205276   
2020-01-08   72.954910   94.598503   69.634544  115.165955  213.346481   
...                ...         ...         ...         ...         ...   
2026-07-27  336.619690  231.389999  326.559998  356.200012  593.869995   
2026-07-28  339.786926  230.860001  333.709991  357.309998  593.409973   
2026-07-29  337.898590  226.649994  336.709991  344.709991  585.609985   
2026-07-30  333.142670  235.500000  333.660004  350.850006  539.030029   
2026-07-31  308.643829  271.579987  356.130005  351.790009  556.710022   

                  MSFT        NVDA   

Adjust prices for splits and dividends

In [57]:
testing_phase_1 = yf.download(
    "AAPL",
    start="2020-01-01",
    end="2026-08-01",
    auto_adjust=True
)

print(testing_phase_1.head())

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.271553  72.331709  71.029930  71.282583  135480400
2020-01-03  71.568924  72.326889  71.345145  71.501549  146322800
2020-01-06  72.139198  72.177699  70.442799  70.693051  118387200
2020-01-07  71.799934  72.403897  71.580965  72.148835  108872000
2020-01-08  72.954910  73.255691  71.503945  71.503945  132079200


Check for missing values

In [58]:
print(prices.isna().sum())
print(prices.isna().sum().sum())

AAPL     0
AMZN     0
GOOGL    0
JPM      0
META     0
MSFT     0
NVDA     0
TSLA     0
UNH      0
XOM      0
dtype: int64
0


Align trading dates

In [59]:
prices = prices.dropna()
print(prices.isna().sum())

AAPL     0
AMZN     0
GOOGL    0
JPM      0
META     0
MSFT     0
NVDA     0
TSLA     0
UNH      0
XOM      0
dtype: int64


if there are only a few missing prices

In [60]:
# ffill() is a method in pandas that fills missing values in a DataFrame or Series by propagating the last valid observation forward. This means that if there are any NaN (Not a Number) values in the data, they will be replaced with the most recent non-NaN value that precedes them. This is particularly useful in time series data where you want to maintain continuity and avoid gaps caused by missing data points.

prices = prices.ffill()
prices = prices.dropna()

Remove stale prices

In [61]:
unchanged = prices.pct_change() == 0

for stock in prices.columns:
    zero_returns = (prices[stock].pct_change() == 0).sum()
    print(stock, zero_returns)

AAPL 4
AMZN 3
GOOGL 2
JPM 3
META 2
MSFT 2
NVDA 2
TSLA 1
UNH 1
XOM 3


Convert prices to log returns

In [62]:
log_returns = np.log(prices / prices.shift(1))
log_returns = log_returns.dropna()

print(log_returns.head())

                AAPL      AMZN     GOOGL       JPM      META      MSFT  \
Date                                                                     
2020-01-03 -0.009770 -0.012213 -0.005245 -0.013284 -0.005305 -0.012530   
2020-01-06  0.007937  0.014776  0.026305 -0.000795  0.018658  0.002581   
2020-01-07 -0.004715  0.002089 -0.001933 -0.017147  0.002161 -0.009160   
2020-01-08  0.015958 -0.007839  0.007093  0.007771  0.010087  0.015803   
2020-01-09  0.021018  0.004788  0.010443  0.003645  0.014210  0.012415   

                NVDA      TSLA       UNH       XOM  
Date                                                
2020-01-03 -0.016136  0.029203 -0.010171 -0.008072  
2020-01-06  0.004185  0.019072  0.006918  0.007649  
2020-01-07  0.012034  0.038067 -0.006055 -0.008218  
2020-01-08  0.001874  0.048033  0.020865 -0.015195  
2020-01-09  0.010922 -0.022189 -0.005694  0.007627  


Winsorize extreme returns

In [63]:
lower = log_returns.quantile(0.01)
upper = log_returns.quantile(0.99)

clean_returns = log_returns.clip(
    lower=lower,
    upper=upper,
    axis=1
)

In [64]:
USE_WINSORIZATION = False

In [65]:
if USE_WINSORIZATION:

    lower = log_returns.quantile(0.01)
    upper = log_returns.quantile(0.99)

    clean_returns = log_returns.clip(
        lower=lower,
        upper=upper,
        axis=1
    )

else:
    clean_returns = log_returns.copy()

Create portfolio weights

In [66]:
weights = {
    "AAPL": 0.10,
    "MSFT": 0.10,
    "NVDA": 0.10,
    "JPM": 0.10,
    "XOM": 0.10,
    "AMZN": 0.10,
    "META": 0.10,
    "GOOGL": 0.10,
    "TSLA": 0.10,
    "UNH": 0.10
}

In [67]:
print(sum(weights.values()))

1.0


Build weighted portfolio returns

In [68]:
# .mul() is a method in pandas that performs element-wise multiplication between two DataFrames or a DataFrame and a Series. In this case, it multiplies the clean_returns DataFrame by the weights_series, aligning them by their index (dates) and columns (stock tickers). The axis=1 argument specifies that the multiplication should be done along the columns (i.e., for each stock).
# .sum(axis=1) is a method that sums the values along the specified axis. In this case, axis=1 means that it sums across the columns (i.e., for each date), resulting in a single Series of portfolio returns for each date.


weights_series = pd.Series(weights)

portfolio_returns = clean_returns.mul(
    weights_series,
    axis=1
).sum(axis=1)

Calculate rolling volatility

In [69]:
rolling_volatility = portfolio_returns.rolling(
    window=30
).std()

In [70]:
rolling_volatility = (
    portfolio_returns
    .rolling(window=30)
    .std()
    * np.sqrt(252)
)

 DISPLAY RESULTS

In [71]:
print("\nClean Prices:")
print(prices.head())

print("\nClean Returns:")
print(clean_returns.head())

print("\nPortfolio Returns:")
print(portfolio_returns.head())

print("\nRolling Volatility:")
print(rolling_volatility.head(35))


Clean Prices:
                 AAPL       AMZN      GOOGL         JPM        META  \
Date                                                                  
2020-01-02  72.271545  94.900497  67.832504  117.899193  207.953827   
2020-01-03  71.568916  93.748497  67.477661  116.343338  206.853500   
2020-01-06  72.139206  95.143997  69.276215  116.250832  210.749298   
2020-01-07  71.799904  95.343002  69.142403  114.274513  211.205276   
2020-01-08  72.954910  94.598503  69.634544  115.165955  213.346481   

                  MSFT      NVDA       TSLA         UNH        XOM  
Date                                                                
2020-01-02  151.544266  5.963803  28.684000  261.702148  52.606392  
2020-01-03  149.657288  5.868346  29.534000  259.053894  52.183475  
2020-01-06  150.044113  5.892957  30.102667  260.852173  52.584145  
2020-01-07  148.676041  5.964301  31.270666  259.277496  52.153778  
2020-01-08  151.044220  5.975488  32.809334  264.744171  51.367287  

Cle

In [72]:
prices.to_csv("../data/processed/clean_prices.csv")

In [73]:
clean_returns.to_csv("../data/processed/clean_returns.csv")

In [74]:
portfolio_returns.to_csv(
    "../data/processed/portfolio_returns.csv",
    header=True
)

In [75]:
rolling_volatility.to_csv(
    "../data/processed/rolling_volatility.csv",
    header=True
)

validation checks

In [76]:
print("========== VALIDATION ==========")

print("Price shape:", prices.shape)

print("Return shape:", clean_returns.shape)

print(
    "Missing prices:",
    prices.isna().sum().sum()
)

print(
    "Missing returns:",
    clean_returns.isna().sum().sum()
)

print(
    "Portfolio return observations:",
    len(portfolio_returns)
)

print(
    "Weight sum:",
    # weights.sum()
)

print(
    "Minimum price:",
    prices.min().min()
)

print(
    "Maximum price:",
    prices.max().max()
)

========== VALIDATION ==========
Price shape: (1653, 10)
Return shape: (1652, 10)
Missing prices: 0
Missing returns: 0
Portfolio return observations: 1652
Weight sum:
Minimum price: 4.885129451751709
Maximum price: 787.419189453125
